# ViRead VietOCR V2 — Flow 2 Google Drive-First Colab Free Worker (0 VNĐ)

Dùng notebook này trên **Google Colab Free (GPU T4)** để chạy **tối đa một job huấn luyện hoặc đánh giá READY** từ hàng đợi ViRead Flow 2.

**Chỉ cần nhập duy nhất 1 token (`VIREAD_TRAINING_WORKER_TOKEN` hoặc `IREAD_TRAINING_WORKER_TOKEN`):**
- Bạn có thể thêm vào **Colab Secrets** (biểu tượng chìa khóa bên trái) với tên `VIREAD_TRAINING_WORKER_TOKEN` (hoặc `IREAD_TRAINING_WORKER_TOKEN`), **hoặc** chạy thẳng cell bên dưới và nhập token vào ô ẩn khi được hỏi.
- `VIREAD_API_BASE` đã được mặc định sẵn là `https://capstone1-ssd8.onrender.com` (không cần cấu hình trừ khi muốn ghi đè).
- **Hỗ trợ đổi tài khoản Google Colab thoải mái (không cần `drive.mount`):** Toàn bộ dữ liệu đọc/ghi — gồm ảnh crop, dataset snapshot, trạng thái **resume giữa epoch**, checkpoint và báo cáo benchmark — đều đi qua Backend ViRead được xác thực bằng worker token và lưu **100% vào một Google Drive chủ sở hữu duy nhất** đã kết nối trong ViRead Admin (ví dụ `cuongtran290605@gmail.com`). Dù bạn đổi sang tài khoản Google Colab khác để lấy thêm quota GPU T4 miễn phí, không có dữ liệu nào rơi sang Drive tài khoản phụ và phiên Colab mới vẫn tự resume đúng giữa epoch.
- Không có OAuth token Google Drive nào được gửi vào Colab. Mọi tệp tải xuống/tải lên đều xác minh SHA-256 chặt chẽ.
- Model đang hoạt động **không bao giờ tự kích hoạt sau khi train xong** — bắt buộc phải vượt Locked Test + Quality Gate (`V2_WIN`) và được Admin phê duyệt.

Trong Colab (bằng bất kỳ tài khoản Google nào), chọn **Runtime → Change runtime type → T4 GPU**, nhập `VIREAD_TRAINING_WORKER_TOKEN` (hoặc `IREAD_TRAINING_WORKER_TOKEN`), sau đó bấm **Runtime → Run all**.


In [ ]:
from getpass import getpass
from pathlib import Path
import os, re, shutil, subprocess, sys

try:
    from google.colab import userdata
except Exception:
    userdata = None

DEFAULT_API_BASE = "https://capstone1-ssd8.onrender.com"
WORKER_TOKEN_NAMES = ("VIREAD_TRAINING_WORKER_TOKEN", "IREAD_TRAINING_WORKER_TOKEN")

def secret(*names, required=True, default=None, prompt=None):
    for name in names:
        value = None
        if userdata is not None:
            try:
                value = userdata.get(name)
            except Exception:
                value = None
        if not value:
            value = os.environ.get(name)
        if value and str(value).strip():
            return str(value).strip()
    if default is not None and str(default).strip():
        return str(default).strip()
    if prompt:
        entered = getpass(prompt).strip()
        if entered:
            return entered
    if required:
        joined = " / ".join(names)
        raise RuntimeError(f"Missing Colab Secret or input: {joined}")
    return None

API_BASE = secret("VIREAD_API_BASE", "IREAD_API_BASE", required=False, default=DEFAULT_API_BASE).rstrip("/")
WORKER_TOKEN = secret(
    *WORKER_TOKEN_NAMES,
    required=True,
    prompt="Nhập VIREAD_TRAINING_WORKER_TOKEN (hoặc IREAD_TRAINING_WORKER_TOKEN): ",
)
RAW_REPOSITORY_REF = (secret("VIREAD_REPOSITORY_REF", required=False) or "").strip()
if RAW_REPOSITORY_REF and RAW_REPOSITORY_REF != "main":
    print(f"Ignoring stale Colab Secret VIREAD_REPOSITORY_REF={RAW_REPOSITORY_REF!r}; syncing to production branch 'main'.")
REPOSITORY_REF = "main"
REPOSITORY_URL = "https://github.com/trancuong3/capstone1.git"
REPO = Path("/content/viread-capstone1")

if REPO.exists():
    status = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain"], capture_output=True, text=True)
    if status.returncode != 0 or status.stdout.strip():
        shutil.rmtree(REPO, ignore_errors=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REPOSITORY_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "--force", "FETCH_HEAD"], check=True)
checked_out = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Synced ViRead worker repo to {REPOSITORY_REF} @ {checked_out} (API_BASE={API_BASE})")
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(REPO / "ai/vietocr_v2/requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-deps", "vietocr==0.3.12"], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for mod in [m for m in list(sys.modules) if m == "ai" or m.startswith("ai.")]:
    sys.modules.pop(mod, None)
os.chdir(REPO)
from ai.vietocr_v2.colab_worker import run_next_job_once
from ai.vietocr_v2.train import prepare_vietocr_training_runtime

runtime = prepare_vietocr_training_runtime()
print(f"Training dependencies ready: NumPy {runtime['numpy_version']}, Pillow {runtime['pillow_version']}, prefetch-generator {runtime['prefetch_generator_version']}, PyTorch {runtime['torch_version']}, torchvision {runtime['torchvision_version']}, CUDA available={runtime['cuda_available']}")

LOCAL_WORK = Path("/content/viread-vietocr-work")
LOCAL_WORK.mkdir(parents=True, exist_ok=True)

print("Checking for one ready Flow 2 training/evaluation job (all data & resume synced directly to owner Google Drive via ViRead backend).")
processed = run_next_job_once(
    API_BASE,
    WORKER_TOKEN,
    LOCAL_WORK,
    use_remote_resume_store=True,
)
if processed:
    print("One job finished processing. Check ViRead Admin for run, benchmark, quality gate, and approval controls.")
else:
    print("No ready job was available. No training run was started.")
